# Agentic RL 引入

## 引入

### 从 Prompt Engineering 到 Agentic RL

| 维度 | Prompt Engineering | Agentic-RL |
|:---|:---|:---|
| 能力来源 | 基座模型预训练知识 + 提示词引导 | 基座模型 + 任务特定的 RL 优化 |
| 开发成本 | 低（工程师时间） | 高（GPU 算力 + 数据标注） |
| 任务适应性 | 通用但不精专 | 针对特定任务深度优化 |
| 推理效率 | 依赖长 prompt，Token 消耗大 | 能力内化到权重，推理更高效 |
| 可扩展性 | 受限于上下文窗口和 prompt 长度 | 可通过持续训练迭代提升 |
| 能力上界 | 受限于基座模型 | 可超越基座模型（涌现能力） |
| 适用场景 | 快速原型、通用任务、低频需求 | 高频、高价值、有明确评估标准的任务 |


### 何时应当选择 Agentic RL？

**适合投入 Agentic RL 的场景**：
- 任务具有客观可验证的评估标准（代码测试通过率、数学答案正确性、API 调用成功率）
- 任务高频重复，训练成本可被长期收益摊薄
- 当前基座模型在该任务上存在系统性、可改进的缺陷
- 具备足够的训练数据或可通过自动化方式生成数据

**不适合 Agentic-RL 的场景**：
- 一次性、低频的开放式任务（ROI 不足）
- 无法客观量化评估的任务（如开放式创意写作）
- 基座模型 + prompt 方式已达到可接受水平
- 缺乏 GPU 算力资源（7B 模型 GRPO 训练至少需要 1× A100 40GB）

### Agentic RL 的理论基础：MDP 框架

马尔可夫决策过程（MDP）
马尔可夫决策过程（MDP）是一种用于描述和解决决策问题的数学框架。它由状态空间、动作空间、奖励函数和转移概率组成：
$$
\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, \mathcal{T}, \mathcal{R}, \gamma \rangle
$$

| MDP 要素 | 形式化定义 | Agent 场景中的对应 |
|:---|:---|:---|
| 状态空间 S | $s_t \in S$ | 当前对话历史 + 工具返回结果 + 环境上下文 |
| 动作空间 A | $a_t \in A$ | 模型的下一次 Token 序列输出（文本、工具调用、代码等） |
| 转移函数 T | $s_{t+1} \sim T(\cdot \mid s_t, a_t)$ | 环境对动作的响应（工具执行结果、用户反馈） |
| 奖励函数 R | $r_t = R(s_t, a_t)$ | 任务完成度评估（答案正确性、代码通过率等） |
| 策略 $\pi_\theta$ | $a_t \sim \pi_\theta(\cdot \mid s_t)$ | 模型参数 $\theta$ 决定的条件生成分布 |
| 折扣因子 $\gamma$ | $\gamma \in [0,1]$ | 对未来奖励的折扣（通常取 1.0，即不折扣） |


补充说明：
- 动作空间：你可以做什么（所有选项的菜单）
- 策略：你实际上会怎么选（决策规则）
- 折扣因子 $\gamma$ ：未来的奖励，打几折？
    - $\gamma = 0$ ：只看眼前，不管未来（近视眼）
    - $\gamma = 1$ ：未来的奖励和当下的奖励一样重要（远视眼）
    - $\gamma = 0.9$ ：明天的 1 块钱，今天看来只值 9 毛


训练的目标是最大化期望累计奖励：
$$
\theta^{*}=
\arg\max_{\theta}
\mathbb{E}_{\tau \sim \pi_{\theta}}
\left[
\sum_{t=0}^{T}\gamma^{t}r_{t}
\right]
$$

- $\theta^*$：最优模型参数，即我们希望通过训练找到的目标
- $\arg\max_\theta$：在所有可能的模型参数 $\theta$ 中，找到使目标函数最大的那个
- $\mathbb{E}_{\tau \sim \pi_\theta}[\cdot]$：期望运算符——由于模型生成是随机的（temperature > 0），同一问题每次生成的轨迹 $\tau$ 不同，我们优化的是所有可能轨迹上的平均表现，而非某一次特定生成的结果
- $\pi_\theta$：由参数 $\theta$ 决定的策略，即"遇到某种情况时怎么行动"。
- $\tau \sim \pi_\theta$：Agent 按该策略完成一次任务，得到一条完整轨迹 $\tau$。
- $\sum_{t=0}^{T} \gamma^t r_t$：折扣累积奖励，将轨迹中每一步的即时奖励 $r_t$ 按时间折扣 $\gamma^t$ 加权求和；$\gamma<1$ 时，近期奖励比远期奖励权重更大（在 Agentic-RL 中通常取 $\gamma=1.0$，即不折扣，因为我们关心任务的最终完成质量而非中间步骤的时序差异）
- $\tau=(s_0, a_0, s_1, a_1, \ldots, s_T)$：一条完整的交互轨迹，记录了从初始状态到终止状态的完整状态-动作序列

### Agentic RL 训练框架总览

<p align="center">
  <img src="./img/agentic_rl_overview.svg" alt="Agentic RL Overview">
</p>


## SFT → RL 两阶段范式

当代主流的 Agentic-RL 训练遵循 SFT → RL 的两阶段范式。

### SFT 阶段

SFT 的本质是"模仿学习"——给模型看大量专家示范，让模型学会"在这种输入下，专家会输出什么"。这个阶段类似于"临帖练字"——先模仿正确的行为模式，建立格式规范和基本能力。

$$
\mathcal{L}_{\mathrm{SFT}}(\theta)=
-\mathbb{E}_{(x,y^*)\sim\mathcal{D}_{\mathrm{SFT}}}
\left[
\log \pi_\theta(y^* \mid x)
\right]
$$

- $-\mathbb{E}_{(x,y^*) \sim D_{\text{SFT}}}$：负号 + 期望符号，表示在监督微调数据集 $D_{\text{SFT}}$ 上，对所有样本 $(x,y^*)$ 取平均。
- $x$：模型的输入（如用户指令、提示词）。
- $y^*$：对应的人类标注的理想输出（即"标准答案"）。
- $D_{\text{SFT}}$：收集好的指令 - 答案对数据集。
- $D_{\text{SFT}} = \{(x^{(i)}, y^{*(i)})\}_{i=1}^{N}$：高质量的 Agent 交互轨迹数据集，每条样本包含输入上下文 $x$（系统提示 + 用户问题）和专家示范输出 $y^*$（含推理过程和工具调用）
- $\log \pi_\theta(y^* \mid x)$：模型在给定输入 $x$ 的条件下，生成专家示范序列 $y^*$ 的对数概率。这个值越大（越接近 0），说明模型认为专家示范是"合理的输出"；加负号后变为损失，最小化损失即最大化对数概率

实际计算时，$\log \pi_\theta(y^* \mid x)$ 按自回归分解展开为逐 token 的对数概率之和：$\sum_{t=1}^{|y^*|} \log \pi_\theta(y_t^* \mid x, y_{<t}^*)$，即每个 token 的生成概率都以前面所有 token 为条件


### RL 阶段

在 $π_{\text{SFT}}$ 的基础上，通过奖励信号引导策略向 $π^∗$ 逼近，突破 SFT 数据的质量上界。
- SFT 的上限：老师什么水平，学生就模仿什么水平，不能超过老师的能力。
- RL 的不同：不要求模仿，只看结果好不好。
- 但 RL 只能突破 SFT 示范数据 的上限，不能自动突破 **奖励函数和评估环境** 的上限。

RL 阶段的训练目标是最大化期望奖励，同时通过 KL 散度约束防止策略偏离过远：
$$
\mathcal{L}_{\mathrm{RL}}(\theta)=
-\mathbb{E}_{\tau \sim \pi_\theta}
\left[R(\tau)\right]
+
\beta \cdot
D_{\mathrm{KL}}
\left(
\pi_\theta \,\|\, \pi_{\mathrm{SFT}}
\right)
$$

第一项：最大化期望奖励
$$
-\mathbb{E}_{\tau \sim \pi_\theta}
\left[R(\tau)\right]
$$

- $\pi_\theta$：当前正在训练的策略。
- $\tau$：模型按照当前策略完成任务所产生的一整条轨迹，例如多轮思考、工具调用和最终回答。
- $R(\tau)$：这条轨迹获得的总奖励。
- $\mathbb{E}$：对多次执行结果计算期望值。
- 负号：训练通常使用"最小化损失"，因此最大化奖励被写成最小化负奖励。（我们希望这个值尽量小，即奖励越大越好）

第二项：限制策略相对 **SFT 策略** 的偏移（KL 散度）
$$
\beta \cdot
D_{\mathrm{KL}}
\left(
\pi_\theta \,\|\, \pi_{\mathrm{SFT}}
\right)
$$

KL 散度衡量当前策略 $\pi_\theta$ 与 SFT 策略 $\pi_{\text{SFT}}$ 的差异：
- 两个策略越接近，KL 散度越小；
- 当前策略偏离 SFT 策略越远，KL 散度越大；
- $\beta$ 控制对这种偏离的惩罚力度。

#### KL 散度
https://haozhe-xing.github.io/agent_learning/zh/appendix/kl_divergence.html

想象你是一名气象预报员。你建立了一个天气预测模型 $Q$，而真实天气的分布是 $P$。KL 散度 $D_{\text{KL}}(P \parallel Q)$ 衡量的是：当你用模型 $Q$ 来近似真实分布 $P$ 时，平均会损失多少信息。

- $D_{KL}(P\|Q)=0$：当且仅当 P 和 Q 完全相同时成立。两个分布越"像"，KL 散度越小。
- $D_{KL}(P\|Q)\geq 0$：KL 散度永远非负（由 Gibbs 不等式保证）。
- $D_{KL}(P\|Q) \neq D_{KL}(Q\|P)$：不对称性！从 P 看 Q 的"距离"和从 Q 看 P 的"距离"通常不同。这就是为什么 KL 散度不是严格意义上的"度量"（metric），而是一种"散度"（divergence）。

KL 散度度量两个概率分布之间的"距离"——但这是一种不对称的距离。

离散情形：
$$
\begin{aligned}
D_{\mathrm{KL}}(P \parallel Q)
&=
\sum_{x \in \mathcal{X}}
P(x)\log\frac{P(x)}{Q(x)} \\
&=
\sum_{x \in \mathcal{X}}
P(x)\left[\log P(x)-\log Q(x)\right]
\end{aligned}
$$

- $P(x)$：真实分布（或目标分布）中事件 $x$ 的概率（权重）
- $Q(x)$：近似分布（或当前模型）中事件 $x$ 的概率（权重）
- $\log P(x) - \log Q(x)$：真实分布与近似分布在事件 $x$ 上的"信息差"
- 整体是一个加权平均：用真实分布 $P$ 作为权重，对每个事件的信息差求期望


连续情形：
$$
D_{\mathrm{KL}}(P \parallel Q)=
\int_{-\infty}^{+\infty}
p(x)\log\frac{p(x)}{q(x)}\,\mathrm{d}x
$$

[todo] KL散度的数学原理

## 数据准备

<p align="center">
  <img src="./img/data_pipeline.svg" alt="Data Pipeline">
</p>


| 阶段 | 数据类型 | 典型规模 | 数据特点 | 核心目标 |
|------|----------|----------|----------|----------|
| 预训练 | 无标注文本（网页、书籍、代码） | 1T—15T tokens | 量大、质参差、覆盖面广 | 学习语言能力和世界知识 |
| SFT（监督微调） | 指令-回答对 / 多轮对话 | 1K—100K 条 | 量少但质极高、格式严格 | 学习行为格式和任务能力 |
| RL（强化学习） | 问题 + 奖励信号 | 10K—1M 条问题 | 需要可评估的奖励函数 | 超越模仿、涌现新策略 |


### SFT 数据
SFT 阶段最重要的发现来自 LIMA [1]（Less Is More for Alignment）论文：<br>
仅用 1,000 条精心筛选的高质量数据，就能让 65B LLaMA 达到接近 GPT-4 的对话质量。

为什么少量数据就够了？
- SFT 不是让模型学习"新知识"——预训练阶段已经具备了丰富的世界知识和语言能力。SFT 的本质是教会模型正确的"行为格式"：

推荐数据量（经验值）：
| 场景 | 推荐数据量 | 说明 |
|------|-----------|------|
| 概念验证 / 研究实验 | 500—1,000 条 | LIMA 规模，验证可行性足够 |
| 单一任务微调（如数学、编码） | 1,000—5,000 条 | 覆盖任务的各种难度和边界情况 |
| Agent 行为格式训练 | 500—2,000 条 | 教会模型 `<think>`/`<tool_call>` 格式 |
| 通用指令遵循 | 5,000—50,000 条 | 覆盖多种任务类型和对话风格 |
| 工业级产品对齐 | 50K—500K 条 | 多轮清洗、多维度质量控制 |

高质量 SFT 数据需要在以下五个维度上达标：
| 质量维度 | 定义 | 检测方法 | 典型问题 |
|:---|:---|:---|:---|
| 正确性 | 回答内容事实准确，逻辑无误 | 人工审核 + 自动事实核查 | 数学计算错误、过时信息 |
| 格式一致性 | 遵循统一的输出格式规范 | 正则表达式校验 | `<think>` 标签不配对、JSON 格式错误 |
| 完整性 | 回答完整地覆盖问题的所有方面 | 人工评分 | 只回答了问题的一半 |
| 难度分布 | 简单/中等/困难任务比例合理 | 自动分类 + 统计 | 全是简单问答，缺少复杂推理 |
| 多样性 | 覆盖不同领域、风格、长度 | 去重 + 聚类分析 | 大量重复或高度相似的样本 |


#### SFT 数据制作方法
- 人工标注
- 强模型蒸馏
- 自动化数据筛选

### RL 数据

RL 数据的核心要素：
- 问题/提示
- 奖励函数
- 参考答案

#### 数据的呈现顺序


<p align="center">
  <img src="./img/curriculum.svg" alt="Curriculum Design">
</p>


### 面试高频题
[面试高频题](https://haozhe-xing.github.io/agent_learning/zh/chapter_llm/08_training_data.html#%E9%9D%A2%E8%AF%95%E9%AB%98%E9%A2%91%E9%A2%98)

# Reference
1. [2.8 SFT 与强化学习训练数据准备](https://haozhe-xing.github.io/agent_learning/zh/chapter_llm/08_training_data.html)
2. [10.1 什么是 Agentic-RL](https://haozhe-xing.github.io/agent_learning/zh/chapter_agentic_rl/01_agentic_rl_overview.html)
3. [附录 E：KL 散度详解](https://haozhe-xing.github.io/agent_learning/zh/appendix/kl_divergence.html)